# Hafta 3 · Kübit, Ölçüm ve Bloch Küresi
**Ders:** Kuantum Hesaplama ve Uygulamaları · **Lab süresi:** ~50 dk · **Ortam:** Google Colab

Bu hafta bir kübitin içinde ne olduğunu **yalnızca ölçüm sonuçlarını kullanarak** bulmayı öğreniyoruz: farklı bazlarda ölçüm, Bloch koordinatları, durum tomografisi, shot istatistiği ve güven aralıkları. Sonunda bir oyun var: **gizli durumu bul!**

| Bölüm | Konu | Süre |
|---|---|---|
| 0 | Kurulum ve yardımcı fonksiyonlar | 3 dk |
| A | Tekrar: vektör, \|⟨φ\|q⟩\|², ⟨Z⟩ | 4 dk |
| B | Ölçüm bazları: Z, X, Y (NumPy) | 6 dk |
| C | Donanımda baz değiştirme: `measure_in_basis` (Qiskit) | 7 dk |
| D | Bloch koordinatları ve iki açı (θ, φ) | 6 dk |
| E | Durum tomografisi, shot etkisi, normalize etme | 8 dk |
| F | Standart hata ve %95 güven aralığı | 6 dk |
| G | Çok kübitte tam ve kısmi ölçüm | 6 dk |
| H | Oyun: gizli durumu bul | 4 dk + ödev |
| I | Alıştırmalar (8 adet) | ödev |

**Bit sırası:** Bu derste her zaman Qiskit sırası kullanılır (q₀ sonuç stringinin **en sağında**).

## 0 · Kurulum

In [ ]:
!pip install -q qiskit qiskit-aer pylatexenc

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector, Operator

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
NAVY, BLUE, ORANGE, GRAY = "#1F3A5F", "#2E6DB4", "#D9822B", "#8A94A6"
rng = np.random.default_rng(2026)
r2 = 1/np.sqrt(2)

def state_to_bloch(amps):
    a, b = complex(amps[0]), complex(amps[1])
    n = np.sqrt(abs(a)**2 + abs(b)**2); a, b = a/n, b/n
    return np.array([2*(np.conj(a)*b).real, 2*(np.conj(a)*b).imag, abs(a)**2 - abs(b)**2])

def _sphere(ax):
    ax.set_box_aspect((1,1,1), zoom=1.3); ax.computed_zorder = False
    u, v = np.linspace(0, 2*np.pi, 50), np.linspace(0, np.pi, 25)
    ax.plot_surface(np.outer(np.cos(u), np.sin(v)), np.outer(np.sin(u), np.sin(v)), np.outer(np.ones_like(u), np.cos(v)),
                    color="#EEF2F8", alpha=0.25, linewidth=0, shade=False)
    t = np.linspace(0, 2*np.pi, 200)
    ax.plot(np.cos(t), np.sin(t), 0, color=GRAY, lw=0.8); ax.plot(np.cos(t), 0*t, np.sin(t), color=GRAY, lw=0.5); ax.plot(0*t, np.cos(t), np.sin(t), color=GRAY, lw=0.5)
    for d in [(1,0,0), (0,1,0), (0,0,1)]: ax.plot([-d[0], d[0]], [-d[1], d[1]], [-d[2], d[2]], color=GRAY, lw=0.7, ls="--")
    for p, s in [((0,0,1.22),"|0⟩ (z)"), ((0,0,-1.25),"|1⟩"), ((1.42,0,0),"|+⟩ (x)"), ((-1.32,0,0),"|−⟩"), ((0,1.32,0),"|+i⟩ (y)"), ((0,-1.32,0),"|−i⟩")]:
        ax.text(*p, s, ha="center", va="center", fontsize=9.5, color=NAVY)
    ax.set_xlim(-1.1, 1.1); ax.set_ylim(-1.1, 1.1); ax.set_zlim(-1.1, 1.1); ax.view_init(elev=18, azim=30); ax.set_axis_off()

def plot_bloch(amps_list, titles=None):
    """Kübit durumlarını (genlik vektörleri) yan yana Bloch küresinde çizer."""
    if np.ndim(amps_list) == 1: amps_list = [amps_list]
    plot_bloch_vectors([[state_to_bloch(a)] for a in amps_list], titles)

def plot_bloch_vectors(groups, titles=None, points=None, colors=(BLUE, ORANGE, NAVY)):
    """groups: her küre için Bloch vektörleri listesi. points: her küre için (k,3) nokta bulutu (isteğe bağlı)."""
    k = len(groups); fig = plt.figure(figsize=(3.6*k, 3.8))
    for i, vecs in enumerate(groups):
        ax = fig.add_subplot(1, k, i+1, projection="3d"); _sphere(ax)
        for j, (x, y, z) in enumerate(vecs):
            c = colors[j % len(colors)]
            ax.plot([0, x], [0, y], [0, z], color=c, lw=3); ax.scatter([x], [y], [z], color=c, s=60, depthshade=False)
        if points is not None and points[i] is not None:
            P = np.asarray(points[i]); ax.scatter(P[:,0], P[:,1], P[:,2], color=ORANGE, s=12, depthshade=False)
        if titles: ax.set_title(titles[i], fontsize=11, color=NAVY)
    plt.show()
print("hazır")

---
## A · Tekrar (Hafta 1–2)
- Kübit: `q = [α, β]`, |α|² + |β|² = 1
- Genel olasılık kuralı: q durumu φ olarak bulunur → **P = |⟨φ|q⟩|²** (`abs(np.vdot(phi, q))**2`)
- ⟨Z⟩ = P(0) − P(1) = Bloch küresinin **z** koordinatı

In [ ]:
ket0, ket1 = np.array([1, 0], dtype=complex), np.array([0, 1], dtype=complex)
plus, minus = np.array([r2, r2], dtype=complex), np.array([r2, -r2], dtype=complex)
plus_i, minus_i = np.array([r2, 1j*r2]), np.array([r2, -1j*r2])

q = np.array([0.6, 0.8], dtype=complex)
print("P(0) =", abs(np.vdot(ket0, q))**2, "  P(1) =", abs(np.vdot(ket1, q))**2)
print("⟨Z⟩  =", abs(q[0])**2 - abs(q[1])**2)

# Z ölçümünde ayırt edilemeyen dört durum:
for name, s in [("|+⟩", plus), ("|−⟩", minus), ("|+i⟩", plus_i), ("|−i⟩", minus_i)]:
    print(f"{name:5s} Z olasılıkları: {np.round(abs(s)**2, 3)}   Bloch: {np.round(state_to_bloch(s), 3)}")
plot_bloch([plus, minus, plus_i, minus_i], ["|+⟩", "|−⟩", "|+i⟩", "|−i⟩"])

---
## B · Ölçüm bazları: Z, X, Y
Bir **ölçüm bazı**, iki dik birim vektördür {|b₀⟩, |b₁⟩}. Bu bazda ölçünce: P(0) = |⟨b₀|q⟩|², P(1) = |⟨b₁|q⟩|².

| Baz | Sonuç 0 | Sonuç 1 | Bloch ekseni |
|---|---|---|---|
| Z | \|0⟩ = [1, 0] | \|1⟩ = [0, 1] | z |
| X | \|+⟩ = [1, 1]/√2 | \|−⟩ = [1, −1]/√2 | x |
| Y | \|+i⟩ = [1, i]/√2 | \|−i⟩ = [1, −i]/√2 | y |

**Benzetme:** aynı veriye (vektöre) farklı *görünüm/projeksiyon* ile bakmak. Ama dikkat: ölçüm veriyi **değiştirir**.

In [ ]:
BASES = {"Z": (ket0, ket1), "X": (plus, minus), "Y": (plus_i, minus_i)}

for b, (b0, b1) in BASES.items():
    print(f"{b} bazı: ⟨b0|b1⟩ = {np.vdot(b0, b1):.3f}  (dik mi? {np.isclose(np.vdot(b0, b1), 0)})")

def show_three_bases(q, title):
    fig, axs = plt.subplots(1, 3, figsize=(10, 2.8), sharey=True)
    for ax, (b, (b0, b1)) in zip(axs, BASES.items()):
        p = [abs(np.vdot(b0, q))**2, abs(np.vdot(b1, q))**2]
        ax.bar(["0", "1"], p, color=[BLUE, NAVY], width=0.5); ax.set_ylim(0, 1.1); ax.set_title(f"{b} bazı", color=NAVY)
        for i, v in enumerate(p): ax.text(i, v + 0.03, f"{v:.3f}", ha="center")
    fig.suptitle(title, color=NAVY); plt.tight_layout(); plt.show()

show_three_bases(np.array([0.6, 0.8]), "q = [0.6, 0.8]")                       # Z: .36/.64  X: .98/.02  Y: .5/.5
show_three_bases(np.array([r2, (1+1j)/2]), "q = [1/√2, (1+i)/2]")               # Z: .5/.5  X: .854/.146  Y: .854/.146

---
## C · Donanım yalnızca Z bazında ölçer: baz değiştirme
`measure` her zaman Z bazındadır. Diğer bazlar için ölçümden **önce** bir kapı uygularız:

| İstenen baz | Devre |
|---|---|
| Z | measure |
| X | **H** → measure |
| Y | **S†** → **H** → measure |

Kural: baz değiştirme matrisinin **satırları**, bazın bra'larıdır (⟨b₀|, ⟨b₁|).

In [ ]:
def measure_in_basis(prep, basis, shots=1000, seed=None):
    """prep: 1 kübitlik hazırlık devresi (ölçümsüz). basis: 'Z', 'X' veya 'Y'. Sayım sözlüğü döndürür."""
    qc = QuantumCircuit(1, 1)
    qc.compose(prep, inplace=True)
    if basis == "X":
        qc.h(0)
    elif basis == "Y":
        qc.sdg(0); qc.h(0)              # önce S†, sonra H
    qc.measure(0, 0)
    sim = AerSimulator(seed_simulator=seed)
    return sim.run(transpile(qc, sim), shots=shots).result().get_counts()

# Y bazında ölçüm devresi
demo = QuantumCircuit(1, 1); demo.sdg(0); demo.h(0); demo.measure(0, 0)
demo.draw("mpl")

In [ ]:
prep = QuantumCircuit(1); prep.ry(2*np.arccos(0.6), 0)        # |0⟩ -> [0.6, 0.8]
print("hazırlanan durum:", Statevector(prep).data)
for b in "ZXY":
    c = measure_in_basis(prep, b, shots=2000, seed=7)
    print(f"{b} bazı: {c}  ->  P(0) ≈ {c.get('0', 0)/2000:.3f}")

In [ ]:
# Satırlar = bra'lar testi
qcY = QuantumCircuit(1); qcY.sdg(0); qcY.h(0)
U_y = Operator(qcY).data
print("H·S† =\n", np.round(U_y, 3))
print("1. satır = ⟨+i| ?", np.allclose(U_y[0], plus_i.conj()))
print("2. satır = ⟨−i| ?", np.allclose(U_y[1], minus_i.conj()))

---
## D · Bloch koordinatları ve iki açı (θ, φ)
- x = ⟨X⟩ = P_X(0) − P_X(1), y = ⟨Y⟩, z = ⟨Z⟩ → her koordinat = **2·P(0) − 1**
- Her kübit: **q = [cos(θ/2), e^(iφ)·sin(θ/2)]**, Bloch = (sin θ cos φ, sin θ sin φ, cos θ)
- θ: kuzey kutbundan açı (P(0) = cos²(θ/2)); φ: ekvatorda +x'ten açı (göreli faz). GPS'teki enlem/boylam gibi.

In [ ]:
def bloch_from_state(q):
    a, b = np.asarray(q, dtype=complex)
    return np.array([2*(np.conj(a)*b).real, 2*(np.conj(a)*b).imag, abs(a)**2 - abs(b)**2])

def state_from_angles(theta, phi):
    return np.array([np.cos(theta/2), np.exp(1j*phi)*np.sin(theta/2)])

def bloch_from_angles(theta, phi):
    return np.array([np.sin(theta)*np.cos(phi), np.sin(theta)*np.sin(phi), np.cos(theta)])

q = state_from_angles(np.pi/3, np.pi/4)
print("vektör      :", q)
print("Bloch (vek.) :", bloch_from_state(q))
print("Bloch (açı)  :", bloch_from_angles(np.pi/3, np.pi/4))
# Üç bazdaki olasılıklardan da aynı koordinatlar:
print("2P(0)-1      :", [2*abs(np.vdot(BASES[b][0], q))**2 - 1 for b in "XYZ"])

In [ ]:
# θ taraması (φ=0) ve φ taraması (θ=90°)
ths = np.radians([0, 45, 90, 135, 180]); phs = np.radians([0, 90, 180, 270])
plot_bloch_vectors([[bloch_from_angles(t, 0) for t in ths], [bloch_from_angles(np.pi/2, p) for p in phs]],
                   ["θ değişiyor (φ = 0)", "φ değişiyor (θ = 90°)"], colors=(BLUE,))

---
## E · Durum tomografisi: vektörü ölçümlerden geri kurmak
1. Z, X, Y bazlarında N'er shot ölç → x̂, ŷ, ẑ = (n₀ − n₁)/N
2. ‖r̂‖ > 1 ise **normalize et** (fiziksel olmayan tahmin)
3. θ̂ = arccos(ẑ/‖r̂‖), φ̂ = atan2(ŷ, x̂) → q̂ = [cos(θ̂/2), e^(iφ̂) sin(θ̂/2)]

Kalite ölçüsü: **sadakat** F = |⟨q|q̂⟩|² = (1 + r·r̂)/2.

In [ ]:
def tomography(prep, shots=1000, seed=None):
    r = []
    for i, b in enumerate("XYZ"):
        c = measure_in_basis(prep, b, shots, seed=None if seed is None else seed + i)
        r.append((c.get("0", 0) - c.get("1", 0)) / shots)
    r = np.array(r)
    raw_norm = np.linalg.norm(r)
    if raw_norm > 1:
        r = r / raw_norm
    theta = np.arccos(np.clip(r[2] / max(np.linalg.norm(r), 1e-12), -1, 1))
    phi = np.arctan2(r[1], r[0]) % (2*np.pi)
    return {"bloch": r, "raw_norm": raw_norm, "theta": theta, "phi": phi, "state": state_from_angles(theta, phi)}

def fidelity(q, q_hat):
    q, q_hat = np.asarray(q, complex), np.asarray(q_hat, complex)
    return abs(np.vdot(q/np.linalg.norm(q), q_hat/np.linalg.norm(q_hat)))**2

def prep_from_angles(theta, phi):
    qc = QuantumCircuit(1); qc.ry(theta, 0); qc.rz(phi, 0)   # rz global faz ekler, önemsiz
    return qc

true_t, true_p = np.pi/3, np.pi/4
prep = prep_from_angles(true_t, true_p)
res = tomography(prep, shots=1000, seed=10)
print("tahmin Bloch :", res["bloch"], " ham boy:", round(res["raw_norm"], 4))
print("θ̂, φ̂ (derece):", np.degrees([res["theta"], res["phi"]]), "  gerçek: [60, 45]")
print("sadakat F    :", fidelity(state_from_angles(true_t, true_p), res["state"]))
plot_bloch_vectors([[bloch_from_angles(true_t, true_p), res["bloch"]]], ["mavi: gerçek, turuncu: tahmin"])

### Shot sayısının etkisi: 100 / 1000 / 10000
Aynı durum için tomografiyi 20 kez tekrarlayıp tahmin bulutlarını çiziyoruz (turuncu noktalar).

In [ ]:
r_true = bloch_from_angles(true_t, true_p)
clouds, errs = [], []
for N in [100, 1000, 10000]:
    ests = np.array([tomography(prep, shots=N, seed=1000*N + 10*k)["bloch"] for k in range(20)])
    clouds.append(ests); errs.append(np.mean(np.linalg.norm(ests - r_true, axis=1)))
plot_bloch_vectors([[r_true]]*3, [f"N = {N}: ort. hata {e:.3f}" for N, e in zip([100, 1000, 10000], errs)], points=clouds)
print("Ortalama hata oranları:", np.round(errs, 4), " -> ~1/√N")

In [ ]:
# Tahmin vektörünün boyu 1'i aşabilir (normalize etmeden önce)
def raw_bloch(r, N):          # hızlı simülasyon: her baz bir binom deneyi
    return 2*rng.binomial(N, (1 + r)/2)/N - 1
for N in [100, 1000, 10000]:
    L = np.array([np.linalg.norm(raw_bloch(r_true, N)) for _ in range(3000)])
    print(f"N={N:5d}:  ‖r̂‖ > 1 oranı = {np.mean(L > 1):.2f}   ortalama |‖r̂‖ − 1| = {np.mean(abs(L - 1)):.4f}")

---
## F · Standart hata ve %95 güven aralığı
- SE(p̂) = √(p̂(1 − p̂)/N),  SE(⟨Z⟩) = √((1 − ⟨Z⟩²)/N)
- **%95 GA = tahmin ± 1.96·SE**
- Gerekli shot: N ≥ 1.96²·p(1 − p)/ε²

In [ ]:
# Sayımlardan ⟨Z⟩ ve güven aralığı
c = measure_in_basis(prep_from_angles(2*np.arccos(np.sqrt(0.82)), 0), "Z", shots=1000, seed=3)
n0, n1 = c.get("0", 0), c.get("1", 0); N = n0 + n1
e = (n0 - n1)/N; se = np.sqrt((1 - e**2)/N)
print(c, f" ⟨Z⟩ ≈ {e:.3f} ± {1.96*se:.3f}   (gerçek 0.64)")

# 30 bağımsız deney: aralıkların kaçı gerçeği kapsıyor?
ztrue, N = 0.5, 200
fig, ax = plt.subplots(figsize=(9, 3.4)); hit = 0
for k in range(30):
    zh = 2*rng.binomial(N, (1 + ztrue)/2)/N - 1; se = np.sqrt((1 - zh**2)/N); ok = abs(zh - ztrue) <= 1.96*se; hit += ok
    ax.errorbar(k, zh, yerr=1.96*se, fmt="o", color=BLUE if ok else ORANGE, capsize=3)
ax.axhline(ztrue, color=NAVY, ls="--"); ax.set_title(f"30 deneyin {hit}'i gerçek değeri kapsıyor", color=NAVY); plt.show()

In [ ]:
Ns = np.logspace(1, 5, 100)
plt.figure(figsize=(7, 3.4))
for z0, ls in [(0, "-"), (0.5, "--"), (0.9, ":")]:
    plt.loglog(Ns, 1.96*np.sqrt((1 - z0**2)/Ns), ls, label=f"⟨Z⟩ = {z0}")
plt.xlabel("shot N"); plt.ylabel("%95 GA yarı genişliği"); plt.legend(frameon=False); plt.grid(alpha=0.2, which="both"); plt.show()
for eps in [0.05, 0.03, 0.01]:
    print(f"P(0)'ı ±{eps} için en kötü durumda: {int(np.ceil(1.96**2*0.25/eps**2))} shot")

---
## G · Çok kübitte ölçüm
**Tüm kübitler:** P(i) = |ψᵢ|². **Tek kübit (kısmi ölçüm)** — üç adım:
1. Marjinal olasılık: kübit k'nın biti m olan indekslerin olasılık toplamı
2. Filtrele: biti m olmayan genlikleri sıfırla
3. Yeniden normalize et: √P'ye böl (koşullu olasılık)

In [ ]:
# Bell durumu: iki kübiti birden ölç
bell = QuantumCircuit(2, 2); bell.h(0); bell.cx(0, 1); bell.measure([0, 1], [0, 1])
display(bell.draw("mpl"))
sim = AerSimulator(seed_simulator=5)
print("Bell, tüm kübitler:", sim.run(transpile(bell, sim), shots=1000).result().get_counts())

In [ ]:
def show_partial(psi, k=0, m=0, name=""):
    psi = np.asarray(psi, dtype=complex); idx = np.arange(len(psi))
    mask = ((idx >> k) & 1) == m
    p = np.sum(abs(psi[mask])**2)
    filt = np.where(mask, psi, 0); after = filt/np.sqrt(p)
    print(f"--- {name}: q{k} = {m} ölçüldü")
    for i in idx:
        print(f"  {i:02b}:  {psi[i].real:+.3f} -> {filt[i].real:+.3f} -> {after[i].real:+.3f}")
    print(f"  P(q{k} = {m}) = {p:.3f}")
    return after

prod = np.kron([0.6, 0.8], [r2, r2])               # q1 = [0.6,0.8], q0 = |+⟩
after = show_partial(prod, 0, 0, "Çarpım durumu")
print("  q1 hâlâ [0.6, 0.8] mi?", np.allclose(after, np.kron([0.6, 0.8], [1, 0])))
after = show_partial(np.array([r2, 0, 0, r2]), 0, 0, "Bell durumu")
print("  Sonuç |00⟩: q1 artık kesin 0")
after = show_partial(np.array([0.5, 0.5, 0.1, 0.7]), 0, 0, "Genel durum")

In [ ]:
# Qiskit ile doğrulama: marjinal olasılık ve ölçüm sonrası durum
sv = Statevector(np.array([0.5, 0.5, 0.1, 0.7]))
print("P(q0) marjinal:", sv.probabilities([0]))           # [0.26, 0.74]
print("P(q1) marjinal:", sv.probabilities([1]))           # [0.5, 0.5]
# Devrede yalnızca q0 ölçümü + ardından q1 ölçümü: koşullu olasılıkları sayımlardan görmek
qc = QuantumCircuit(2, 2); qc.initialize([0.5, 0.5, 0.1, 0.7], [0, 1]); qc.measure(0, 0); qc.measure(1, 1)
c = sim.run(transpile(qc, sim), shots=20000).result().get_counts()
n_q0_0 = c.get("00", 0) + c.get("10", 0)
print("P(q1=0 | q0=0) ≈", round(c.get("00", 0)/n_q0_0, 3), "  (teorik 0.962)")

---
## H · Oyun: gizli durumu bul
Notebook rastgele bir **gizli kübit durumu** seçer. Tek erişiminiz `black_box(basis, shots)`: istediğiniz bazda ölçüm yapar, sayımları verir. Toplam **shot bütçesi 3000**. Tahmininizi (θ, φ) olarak verin; puanınız sadakat F ile hesaplanır.

| F | Puan |
|---|---|
| ≥ 0.999 | ★★★ |
| ≥ 0.99 | ★★ |
| ≥ 0.95 | ★ |

In [ ]:
def hidden_state_game(seed=None, budget=3000):
    g = np.random.default_rng(seed)
    # küre üzerinde düzgün rastgele nokta
    theta = np.arccos(1 - 2*g.random()); phi = 2*np.pi*g.random()
    prep = prep_from_angles(theta, phi); used = {"shots": 0}
    def black_box(basis, shots):
        if used["shots"] + shots > budget:
            raise RuntimeError(f"Bütçe aşıldı! Kalan: {budget - used['shots']}")
        used["shots"] += shots
        return measure_in_basis(prep, basis, shots, seed=int(g.integers(1e9)))
    def score(theta_hat, phi_hat):
        F = fidelity(state_from_angles(theta, phi), state_from_angles(theta_hat, phi_hat))
        stars = "★★★" if F >= 0.999 else "★★" if F >= 0.99 else "★" if F >= 0.95 else "tekrar dene"
        print(f"Sadakat F = {F:.5f}  ->  {stars}   (kullanılan shot: {used['shots']})")
        print(f"Gerçek: θ = {np.degrees(theta):.1f}°, φ = {np.degrees(phi):.1f}°   Tahmin: θ = {np.degrees(theta_hat):.1f}°, φ = {np.degrees(phi_hat):.1f}°")
        plot_bloch_vectors([[bloch_from_angles(theta, phi), bloch_from_angles(theta_hat, phi_hat)]], ["mavi: gerçek, turuncu: tahmin"])
        return F
    return black_box, score

black_box, score = hidden_state_game(seed=42)
# Örnek strateji: bütçeyi üç baza eşit dağıt
r = []
for b in "XYZ":
    c = black_box(b, 1000)
    r.append((c.get("0", 0) - c.get("1", 0))/1000)
r = np.array(r); r = r/np.linalg.norm(r)
F = score(np.arccos(r[2]), np.arctan2(r[1], r[0]) % (2*np.pi))

**Siz deneyin:** `hidden_state_game(seed=None)` ile yeni bir gizli durum alın ve farklı stratejiler deneyin (ör. önce 300'er shot ile kaba tahmin, sonra kalan bütçeyi belirsizliği en yüksek eksene harcayın).

---
## I · Alıştırmalar
`# TODO` yerlerini doldurun; `assert` satırları geçerse çözüm doğrudur.

### Alıştırma 1 · Bazda olasılık
`prob_in_basis(q, basis)` fonksiyonu `np.array([P(0), P(1)])` döndürsün (`BASES` sözlüğünü kullanın).

In [ ]:
def prob_in_basis(q, basis):
    # TODO
    pass

assert np.allclose(prob_in_basis([0.6, 0.8], "X"), [0.98, 0.02])
assert np.allclose(prob_in_basis([0.6, 0.8], "Y"), [0.5, 0.5])
assert np.allclose(prob_in_basis([r2, (1+1j)/2], "Y"), [0.8536, 0.1464], atol=1e-4)
assert np.allclose(prob_in_basis(minus_i, "Y"), [0, 1])
print("Alıştırma 1 ✓")

### Alıştırma 2 · Baz değiştirme devresi
`basis_change(basis)` ölçümden önce eklenecek 1 kübitlik devreyi döndürsün. Test: devrenin matrisinin satırları bazın bra'ları olmalı.

In [ ]:
def basis_change(basis):
    qc = QuantumCircuit(1)
    # TODO
    return qc

for b in "ZXY":
    U = Operator(basis_change(b)).data
    b0, b1 = BASES[b]
    assert np.allclose(U[0], b0.conj()) and np.allclose(U[1], b1.conj()), b
print("Alıştırma 2 ✓")

### Alıştırma 3 · Vektörden açılar
`angles_from_state(q)` → `(theta, phi)`, φ ∈ [0, 2π). Global fazı yok sayın (φ = arg β − arg α). |α| ya da |β| ≈ 0 ise φ = 0.

In [ ]:
def angles_from_state(q):
    # TODO
    pass

assert np.allclose(np.degrees(angles_from_state([0.6j, 0.8])), [106.2602, 270], atol=1e-3)
assert np.allclose(angles_from_state([1, 0]), [0, 0])
for _ in range(50):
    t, p = rng.uniform(0.01, np.pi - 0.01), rng.uniform(0, 2*np.pi)
    assert np.allclose(angles_from_state(np.exp(1j*rng.uniform(0, 6))*state_from_angles(t, p)), [t, p])
print("Alıştırma 3 ✓")

### Alıştırma 4 · Sayımlardan Bloch vektörü
`bloch_from_counts(cx, cy, cz)`: üç sayım sözlüğünden (X, Y, Z bazları) `np.array([x, y, z])` döndürsün. Eksik anahtarlara dikkat!

In [ ]:
def bloch_from_counts(cx, cy, cz):
    # TODO
    pass

assert np.allclose(bloch_from_counts({"0": 812, "1": 188}, {"0": 798, "1": 202}, {"0": 752, "1": 248}), [0.624, 0.596, 0.504])
assert np.allclose(bloch_from_counts({"0": 50, "1": 50}, {"0": 50, "1": 50}, {"0": 100}), [0, 0, 1])
print("Alıştırma 4 ✓")

### Alıştırma 5 · Normalize ve sadakat
`normalize_bloch(r)`: boyu 1 yapsın (sıfır vektörde `[0,0,1]`). `fidelity_bloch(r, r_hat)` = (1 + r·r̂)/2. Örnek 9'daki (0.80, 0.04, 0.62) tahminini normalize edin.

In [ ]:
def normalize_bloch(r):
    # TODO
    pass

def fidelity_bloch(r, r_hat):
    # TODO
    pass

rn = normalize_bloch(np.array([0.80, 0.04, 0.62]))
print(rn)
assert np.isclose(np.linalg.norm(rn), 1) and np.allclose(rn, [0.7898, 0.0395, 0.6121], atol=1e-3)
assert np.isclose(fidelity_bloch([0, 0, 1], [0, 0, -1]), 0) and np.isclose(fidelity_bloch([1, 0, 0], [0, 1, 0]), 0.5)
q1, q2 = state_from_angles(1.0, 2.0), state_from_angles(1.3, 2.4)
assert np.isclose(fidelity_bloch(bloch_from_state(q1), bloch_from_state(q2)), fidelity(q1, q2))
print("Alıştırma 5 ✓")

### Alıştırma 6 · %95 güven aralığı ve kapsama oranı
`ci95_expval(counts)` → `(tahmin, alt, üst)`. Sonra 200 bağımsız deneyde (her biri 400 shot, gerçek ⟨Z⟩ = 0.3; sayımları `rng.binomial` ile üretin) aralıkların gerçeği kapsama oranını hesaplayın.

In [ ]:
def ci95_expval(counts):
    # TODO
    pass

assert np.allclose(ci95_expval({"0": 820, "1": 180}), (0.64, 0.5924, 0.6876), atol=1e-4)
coverage = None   # TODO
print("kapsama oranı:", coverage)
assert 0.88 <= coverage <= 1.0
print("Alıştırma 6 ✓")

### Alıştırma 7 · Kaç shot gerekir?
`shots_needed(eps, p=0.5, z=1.96)`: P(0)'ı ±eps hassasiyetle ölçmek için gereken en küçük tam sayı N.

In [ ]:
def shots_needed(eps, p=0.5, z=1.96):
    # TODO
    pass

assert shots_needed(0.01) == 9604 and shots_needed(0.03) == 1068 and shots_needed(0.05) == 385
assert shots_needed(0.01, p=0.1) == 3458
print("Alıştırma 7 ✓")

### Alıştırma 8 · Kısmi ölçüm fonksiyonu
`measure_qubit(state, k, outcome)` → `(olasılık, yeni_durum)`. k = 0 en sağdaki bit (Qiskit sırası). Qiskit'in `Statevector.probabilities([k])` sonucuyla karşılaştırın.

In [ ]:
def measure_qubit(state, k, outcome):
    # TODO
    pass

p, s = measure_qubit([0.5, 0.5, 0.1, 0.7], 0, 0)
assert np.isclose(p, 0.26) and np.allclose(s, [0.9806, 0, 0.1961, 0], atol=1e-4)
p, s = measure_qubit([0, 0.6, 0.8, 0], 1, 1)
assert np.isclose(p, 0.64) and np.allclose(s, [0, 0, 1, 0])
psi = rng.normal(size=8) + 1j*rng.normal(size=8); psi /= np.linalg.norm(psi)
for k in range(3):
    assert np.isclose(measure_qubit(psi, k, 1)[0], Statevector(psi).probabilities([k])[1])
print("Alıştırma 8 ✓")

---
### Haftanın özeti
- Ölçüm bazı = iki dik vektör; P = |⟨b|q⟩|². Z, X, Y bazları Bloch'un üç ekseni
- Donanım yalnızca Z'de ölçer: **X → H**, **Y → S† sonra H**
- Bloch = (⟨X⟩, ⟨Y⟩, ⟨Z⟩); her kübit = [cos(θ/2), e^(iφ) sin(θ/2)]
- Tomografi: 3 baz × N shot → r̂ → normalize → θ̂, φ̂; kalite: sadakat F
- %95 GA = tahmin ± 1.96·SE; hata ~1/√N; N ≥ 1.96²·p(1−p)/ε²
- Kısmi ölçüm = marjinal + filtrele + normalize; çarpım durumunda diğer kübit değişmez, dolanıkta kesinleşebilir

**Gelecek hafta:** Tek kübit kapıları I: X, Y, Z ve H kapılarını Bloch küresinde **tek tek** inceleyeceğiz (her kapı bir eksen etrafında dönüş) ve etkileşimli bir HTML Bloch simülatörü kullanacağız. Bu haftaki `tomography` fonksiyonu, kapıların etkisini ölçümle doğrulamak için tekrar işimize yarayacak.